# Iteration 6: a second matcher (XoFTR) on the photos the 6 queries do not answer

**Loss targeted:** fine matching and the decision on the hard photos (skipped or escalated by today's rule).

**Why now, although Test_Iteration_4 found XoFTR no better than LoFTR overall:** that test replaced LoFTR on every photo and judged both with one
inlier bar. Here XoFTR is only a **second opinion** on the ~10 % of photos the standard queries leave open. XoFTR is trained on thermal-visible
pairs and has a different refinement, so its chance hits on look-alike places should be largely independent of LoFTR's: if both networks put
the 27-query answer at the same place, a false place is unlikely, even when neither reaches its own bar. A matcher's own inlier bar does not
transfer, but agreement between two matchers is a property of the place, not of either count.

**Test.** Rich dumps of the hard photos with XoFTR (MINIMA weights, coarse 0.3, fine 0.1: `Rich_Dump_Run` with `MATCHER=xoftr`). Rules:
(a) today; (b) today + answer an unanswered photo when the LoFTR and XoFTR 27-query choices lie within 1 sigma and both have >= m inliers;
(c) today + XoFTR's 27-query choice at >= b inliers (b chosen on official + NUDT 101). Photos the 6 standard queries answer are unchanged.
**Cost:** a second network on the Jetson (memory: the submitted run peaked at 7.1 of 7.4 GiB) and 27 more queries on these photos.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
pd.set_option("display.width", 220)
from train.checks.replay import load_set, decide_current, pooled, choose, query_ids, WIDE, STANDARD, points, error_sigma, score, paired_bootstrap, query_seconds
DUMPS = ROOT / "tests/Test_Iteration_6/results/rich_dump"
SET_NAMES = {"official": "official", "nudt101": "nudt_seed101", "nudt202": "nudt_seed202", "nudt404": "nudt_fresh404"}
CHOOSE_ON = ["official", "nudt101"]
lo, xo = {}, {}
for n, f in SET_NAMES.items():
    if (DUMPS / f).exists() and (DUMPS / (f + "_xoftr")).exists():
        lo[n] = load_set(DUMPS / f); xo[n] = load_set(DUMPS / (f + "_xoftr"))
print({n: (len(lo[n]), len(xo[n])) for n in lo})

In [ ]:
def choice27(photo):
    c = pooled(photo, query_ids(photo, WIDE)); k = choose(c)
    return (None, None, 0.0) if k < 0 else (c["x"][k], c["y"][k], float(c["inliers"][k]))
rows = []
for n in lo:
    for pid, ph in lo[n].items():
        today = decide_current(ph)
        row = dict(set=n, photo=pid, height=ph["height"], stage=today["stage"], today_points=today["points"], hard=pid in xo[n])
        if pid in xo[n]:
            lx, ly, li = choice27(ph); xx, xy, xi = choice27(xo[n][pid])
            row.update(lo_inl=li, xo_inl=xi, lo_err=error_sigma(ph, lx, ly) if lx is not None else np.nan,
                       xo_err=error_sigma(ph, xx, xy) if xx is not None else np.nan,
                       gap_sigma=np.hypot(lx - xx, ly - xy) / ph["sigma"] if lx is not None and xx is not None else np.inf,
                       xo_seconds=query_seconds(xo[n][pid], query_ids(xo[n][pid], WIDE)), lo_seconds=query_seconds(ph, query_ids(ph, WIDE)))
        rows.append(row)
t = pd.DataFrame(rows)
h = t[t.hard]
print("hard photos (not answered by the 6 queries):", h.groupby("set").size().to_dict())
print("XoFTR 27 queries: %.1f s per photo vs LoFTR %.1f s (PC)" % (h.xo_seconds.mean(), h.lo_seconds.mean()))
display(h.groupby("set").agg(lo_right=("lo_err", lambda s: int((s <= 4).sum())), xo_right=("xo_err", lambda s: int((s <= 4).sum())),
                             both_close=("gap_sigma", lambda s: int((s <= 1).sum()))))

In [ ]:
def rule(t, mode, m=3, b=11):
    p = t.today_points.to_numpy().copy()
    for i, r in enumerate(t.itertuples()):
        if not r.hard or r.stage != "skip":
            continue
        if mode == "agree" and r.gap_sigma <= 1 and r.lo_inl >= m and r.xo_inl >= m:
            p[i] = points(r.lo_err)
        if mode == "xoftr_bar" and r.xo_inl >= b:
            p[i] = points(r.xo_err)
    return p
res = []
for mode, grid in (("agree", [dict(m=m) for m in (1, 2, 3, 4, 5)]), ("xoftr_bar", [dict(b=b) for b in (8, 10, 12, 15, 20)])):
    for kw in grid:
        for n in lo:
            part = t[t.set == n]
            newp = rule(part, mode, **kw)
            g = paired_bootstrap(part.today_points.to_numpy(), newp)
            res.append(dict(mode=mode, setting=str(kw), set=n, today=round(score(part.today_points), 2), new=round(score(newp), 2),
                            gain="%+.2f [%+.2f, %+.2f]" % g, wrong=int((newp < 0).sum()), extra=int((newp != part.today_points.to_numpy()).sum())))
res = pd.DataFrame(res)
display(res.pivot_table(index=["mode", "setting"], columns="set", values=["new", "wrong", "extra"], aggfunc="first"))
display(res.pivot(index=["mode", "setting"], columns="set", values="gain"))
res.to_csv(ROOT / "tests/Test_Iteration_6/results/it6_xoftr_second_opinion.csv", index=False)